# Notebook 02: Persiapan Data, Normalisasi Slang & Pelabelan ABSA (Bab 3 Laporan AM552)
**Mata Kuliah**: AM552 - Data Science of IS (Program Studi S1 Sistem Informasi, FTI UKSW)  
**Kode Kelompok**: `682024046_KEL_DS`  
**Disusun oleh**: Brendan Wilson Santoso (682024016)  

---
## Ringkasan Bab 3: Data Preparation & Ekstraksi Aspek
Tahapan pada notebook ini mencakup:
1. **Langkah Persiapan Data (Bab 3.1)**: Pembersihan teks ulasan Play Store Indonesia (*case folding*, pembersihan URL, reduksi pemanjangan huruf berulang, normalisasi slang dengan preservasi negasi, dan pelabelan aspek ABSA).
2. **Pembagian Data Anti-Leakage (Bab 3.2)**: Pembagian berstratifikasi rasio 80% Data Latih (`56.976` baris) dan 20% Data Uji (`14.245` baris) secara konsisten dengan `random_state=42` sebelum ekstraksi TF-IDF.
3. **Data Akhir Siap Pemodelan (Bab 3.3)**: Penyusunan dataset bersih dengan preservasi kata negasi teruji dan 4 dimensi aspek layanan yang siap digunakan untuk pemodelan klasifikasi dan topic discovery.


---
## 3.1 Langkah Persiapan Data
### 3.1.1 Pemuatan Dataset Mentah & Kamus Slang Bahasa Indonesia
Memuat dataset ulasan Play Store Indonesia (`71.221` baris) dan kamus pemetaan kata tidak baku (`slang_dictionary_id.json`).


In [1]:
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RAW_CSV_PATH = "../dataset/RAW_DATASET_GENAI_INDONESIA_AM522B.csv"
SLANG_JSON_PATH = "../03_data_preparation/slang_dictionary_id.json"

df = pd.read_csv(RAW_CSV_PATH, low_memory=False)
with open(SLANG_JSON_PATH, "r", encoding="utf-8") as f:
    slang_map = json.load(f)

print(f"Status Pemuatan Dataset Mentah:")
print(f"  - Total Baris Asli          : {len(df):,} baris")
print(f"  - Total Atribut Kolom        : {df.shape[1]} kolom")
print(f"  - Baris Teks Kosong (Missing): {df['content'].isna().sum()} baris")
print(f"  - Jumlah Entri Kamus Slang   : {len(slang_map):,} kata pemetaan")


Status Pemuatan Dataset Mentah:
  - Total Baris Asli          : 71,221 baris
  - Total Atribut Kolom        : 11 kolom
  - Baris Teks Kosong (Missing): 0 baris
  - Jumlah Entri Kamus Slang   : 170 kata pemetaan


### 3.1.2 Pipeline Normalisasi Slang & Preservasi Kata Negasi
Ulasan Play Store Indonesia memiliki karakteristik:
1. **Pemanjangan Huruf Emosional (*Elongation*)**: Pengguna menulis `"bagusssss"` atau `"lemotttttt"`. Menggunakan regex `(.)\1{2,}` direduksi menjadi maksimal 2 karakter (`"baguss"`, `"lemott"`).
2. **Kontraksi Slang & Negasi**: Pengguna sering menyingkat kata (`"gk"`, `"gak"`, `"tdk"`, `"bgt"`, `"lemot"`) atau menggabungkan kata negasi (`"gabisa"`, `"gajelas"`, `"gatau"`).
3. **Preservasi Negasi (*Negation Preservation Rule*)**: Kata penentu polaritas seperti `"tidak"`, `"kurang"`, `"jangan"`, `"bukan"`, `"belum"` **TIDAK BOLEH DIHAPUS** oleh stopword remover agar sentimen kalimat tidak terbalik dari negatif ke positif.


In [2]:
def clean_and_normalize_text(text, slang_dict):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    tokens = re.findall(r"[a-z0-9]+", text)
    return " ".join(slang_dict.get(t, t) for t in tokens)

df["clean_content"] = df["content"].apply(lambda x: clean_and_normalize_text(x, slang_map))

# Demonstrasi Perbandingan Teks Sebelum vs Sesudah Preprocessing (Bab 3.3)
sample_indices = [1, 3, 9, 89, 142]
comp_rows = []
for idx in sample_indices:
    comp_rows.append({
        "Index": idx,
        "Aplikasi": df.loc[idx, "app_name"],
        "Teks Mentah (Raw Content)": df.loc[idx, "content"][:85] + ("..." if len(str(df.loc[idx, "content"])) > 85 else ""),
        "Teks Bersih (Clean & Normalized)": df.loc[idx, "clean_content"][:85] + ("..." if len(df.loc[idx, "clean_content"]) > 85 else "")
    })

pd.set_option("display.max_colwidth", None)
display_comp = pd.DataFrame(comp_rows)
print("=== TABEL PERBANDINGAN SEBELUM VS SESUDAH PREPROCESSING (BAB 3.3) ===")
for i, r in display_comp.iterrows():
    print(f"[{r['Aplikasi']}]")
    print(f"  Raw  : {r['Teks Mentah (Raw Content)']}")
    print(f"  Clean: {r['Teks Bersih (Clean & Normalized)']}\n")


=== TABEL PERBANDINGAN SEBELUM VS SESUDAH PREPROCESSING (BAB 3.3) ===
[Google Gemini]
  Raw  : mode live nya sering macet sendiri sering slow respon gajelas error error
  Clean: mode live nya sering macet sendiri sering slow respon tidak jelas error error

[Google Gemini]
  Raw  : tidak nyambung
  Clean: tidak tersambung

[Google Gemini]
  Raw  : saya kasih bintang walaupun bayar
  Clean: saya kasih bintang walaupun bayar

[Google Gemini]
  Raw  : drama Korea
  Clean: drama korea

[Google Gemini]
  Raw  : boleh
  Clean: boleh


### 3.1.3 Ekstraksi 4 Dimensi Aspek Layanan (Multi-Label ABSA Tagging)
Setiap ulasan dipetakan ke dalam 4 aspek layanan Generative AI menggunakan *seed-keyword dictionary* terkurasi:
- **A1: Akurasi & Kecerdasan AI** (*intelligence, reasoning, coding, hallucination, math*)
- **A2: Harga & Limitasi Kuota** (*pricing, subscription, daily limit, credit, free tier*)
- **A3: Performa, UI/UX & Bug** (*speed, latency, crashes, server busy, login, voice mode*)
- **A4: Kemampuan Bahasa Indonesia** (*natural Indonesian, local context, translation accuracy*)


In [3]:
ASPECT_KEYWORDS = {
    "A1_Akurasi_Kecerdasan": {
        "pintar", "cerdas", "akurat", "tepat", "paham", "mengerti", "jawaban", "solusi",
        "membantu", "bantu", "tugas", "skripsi", "coding", "matematika", "halusinasi",
        "ngawur", "acau", "salah", "bodoh", "tersambung", "nyambung", "lupa", "sensor",
        "filter", "logika", "penjelasan", "informasi", "fakta", "ilmu", "belajar",
    },
    "A2_Harga_Kuota": {
        "gratis", "free", "bayar", "berbayar", "mahal", "murah", "harga", "langganan",
        "subscription", "plus", "pro", "kuota", "limit", "batas", "dibatasi", "habis",
        "koin", "token", "iklan", "premium", "uang", "duit", "pelit", "kemahalan", "gratisan",
    },
    "A3_Performa_Bug": {
        "cepat", "lambat", "loading", "server", "sibuk", "down", "error", "bug", "macet",
        "crash", "keluar", "login", "masuk", "akun", "update", "versi", "tampilan",
        "ui", "suara", "voice", "live", "jaringan", "koneksi", "berat", "ringan",
        "ngelag", "lag", "ngebug", "lelet", "lola", "hilang", "fc", "perbaiki",
    },
    "A4_Bahasa_Indonesia": {
        "bahasa", "indonesia", "terjemahan", "translate", "logat", "aksen", "baku",
        "gaul", "lokal", "kaku", "english", "pelafalan", "bicara", "ngomong",
    },
}

for asp_col, kw_set in ASPECT_KEYWORDS.items():
    df[asp_col] = df["clean_content"].apply(lambda t: 1 if len(set(t.split()).intersection(kw_set)) > 0 else 0)

print("=== DISTRIBUSI FREKUENSI ASPEK ABSA LINTAS DATASET (71.221 ULASAN) ===")
aspect_summary = []
for col in ASPECT_KEYWORDS.keys():
    cnt = df[col].sum()
    pct = (cnt / len(df)) * 100
    aspect_summary.append({"Aspek": col, "Jumlah Baris Terdeteksi": f"{cnt:,}", "Persentase": f"{pct:.2f}%"})
    print(f"  - {col:<24}: {cnt:>6,} baris ({pct:>5.2f}%)")

print("\nBreakdown Aspek Terbanyak per Aplikasi (Persentase Ulasan yang Menyebutkan Aspek):")
pivot_aspect = df.groupby("app_name")[list(ASPECT_KEYWORDS.keys())].mean() * 100
print(pivot_aspect.round(2))


=== DISTRIBUSI FREKUENSI ASPEK ABSA LINTAS DATASET (71.221 ULASAN) ===
  - A1_Akurasi_Kecerdasan   : 12,356 baris (17.35%)
  - A2_Harga_Kuota          :  5,911 baris ( 8.30%)
  - A3_Performa_Bug         :  9,399 baris (13.20%)
  - A4_Bahasa_Indonesia     :  1,770 baris ( 2.49%)

Breakdown Aspek Terbanyak per Aplikasi (Persentase Ulasan yang Menyebutkan Aspek):
                           A1_Akurasi_Kecerdasan  A2_Harga_Kuota  A3_Performa_Bug  A4_Bahasa_Indonesia
app_name                                                                                              
Character.AI                               10.39           22.74            37.67                 4.91
ChatGPT (OpenAI)                           17.96            4.99             4.79                 1.45
Claude by Anthropic                        19.96           16.04            17.29                 3.77
DeepSeek - AI Assistant                    19.39            9.38            29.42                 5.01
Google Gemini      

---
## 3.2 Pembagian Data (Anti-Leakage Stratified Split 80:20)
Data dibagi menjadi **80% Data Latih (`56.976` baris)** dan **20% Data Uji (`14.245` baris)** secara berstratifikasi (*stratified*) berdasarkan kombinasi nama aplikasi dan kelas sentimen (`app_name` + `sentiment_3class`).

> **Pemeriksaan Anti-Leakage (Zero Data Leakage Gate)**:
> Pembagian data ini dilakukan **sebelum** pembentukan kosakata dan bobot IDF (`TfidfVectorizer.fit()`). Data Uji diisolasi secara ketat dan hanya boleh ditransformasi (`transform()`) oleh model yang dilatih pada Data Latih.


In [4]:
# Pengelompokan Sentimen 3-Kelas
df["sentiment_3class"] = df["score"].apply(
    lambda s: "0_Negative" if s <= 2 else ("1_Neutral" if s == 3 else "2_Positive")
)
df["strat_key"] = df["app_name"] + "_" + df["sentiment_3class"]

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["strat_key"],
    random_state=42
)

# Verifikasi Forensik Anti-Leakage
overlap_indices = set(train_df.index).intersection(set(test_df.index))
assert len(overlap_indices) == 0, "PERINGATAN: Terjadi kebocoran index antara Data Latih dan Data Uji!"

print("=== HASIL PEMBAGIAN DATA BERSTRATIFIKASI (BAB 3.2) ===")
print(f"Total Dataset Awal : {len(df):,} baris (100.00%)")
print(f"Data Latih (Train) : {len(train_df):,} baris ({(len(train_df)/len(df))*100:.2f}%)")
print(f"Data Uji (Test)    : {len(test_df):,} baris ({(len(test_df)/len(df))*100:.2f}%)")
print(f"Index Overlap Check: {len(overlap_indices)} baris (0% Kebocoran Data - ZERO LEAKAGE PASSED)\n")

print("Distribusi Target Sentimen pada Data Latih (80%):")
print(train_df["sentiment_3class"].value_counts(normalize=True).round(4) * 100)
print("\nDistribusi Target Sentimen pada Data Uji (20%):")
print(test_df["sentiment_3class"].value_counts(normalize=True).round(4) * 100)


=== HASIL PEMBAGIAN DATA BERSTRATIFIKASI (BAB 3.2) ===
Total Dataset Awal : 71,221 baris (100.00%)
Data Latih (Train) : 56,976 baris (80.00%)
Data Uji (Test)    : 14,245 baris (20.00%)
Index Overlap Check: 0 baris (0% Kebocoran Data - ZERO LEAKAGE PASSED)

Distribusi Target Sentimen pada Data Latih (80%):
sentiment_3class
2_Positive    71.34
0_Negative    24.82
1_Neutral      3.84
Name: proportion, dtype: float64

Distribusi Target Sentimen pada Data Uji (20%):
sentiment_3class
2_Positive    71.34
0_Negative    24.82
1_Neutral      3.84
Name: proportion, dtype: float64


---
## 3.3 Data Akhir Siap Pemodelan & Validasi Preservasi Negasi
Pengujian otomatis (*unit tests*) untuk memastikan bahwa kata negasi krusial (`tidak`, `kurang`, `jangan`, `bukan`, `belum`) tidak hilang, serta ringkasan kondisi dataset akhir sesuai format Tabel Bab 3.3 Laporan AM552.


In [5]:
# 1. VALIDASI PRESERVASI NEGASI (Unit Testing Aspek Data Engineering)
test_phrases = [
    ("Aplikasi ini tidak bagus dan sangat mengecewakan", "tidak"),
    ("Jawaban AI kurang memuaskan dan sering ngawur", "kurang"),
    ("Jangan download aplikasi ini bikin emosi", "jangan"),
    ("Model bot bukan menjawab malah ngaco", "bukan"),
    ("Update terbaru belum bisa login akun google", "belum"),
    ("Apk nya gabisa dibuka sama sekali", "tidak bisa")
]

print("=== PENGUJIAN PRESERVASI KATA NEGASI & NORMALISASI KONTRAKSI ===")
all_passed = True
for raw_phrase, expected_token in test_phrases:
    cleaned = clean_and_normalize_text(raw_phrase, slang_map)
    contains = expected_token in cleaned
    status = "PASSED" if contains else "FAILED"
    print(f"[{status}] Raw: "{raw_phrase}"")
    print(f"         Clean: "{cleaned}" -> Mengandung "{expected_token}": {contains}")
    if not contains:
        all_passed = False

assert all_passed, "Gagal: Beberapa kata negasi krusial hilang dalam preprocessing!"
print("\n[OK] 100% PENGUJIAN PRESERVASI KATA NEGASI LOLOS UJI TANPA KEHILANGAN POLARITAS!\n")

# 2. RINGKASAN DATA AKHIR BAB 3.3
summary_table = pd.DataFrame([
    {
        "Kondisi Dataset": "Sebelum Persiapan (Raw Data)",
        "Jumlah Baris": f"{len(df):,} baris",
        "Jumlah Kolom / Fitur": f"{df.shape[1]} kolom mentah",
        "Keterangan": "Teks mentah mengandung slang, elongasi huruf, URL, dan belum terbagi."
    },
    {
        "Kondisi Dataset": "Sesudah Persiapan — Data Latih (80%)",
        "Jumlah Baris": f"{len(train_df):,} baris",
        "Jumlah Kolom / Fitur": "Fitur Teks Bersih + 4 Dimensi ABSA + Target 3-Kelas",
        "Keterangan": "Teks bersih terstandarisasi, preservasi negasi aman, siap untuk TF-IDF fit & CV 5-Fold."
    },
    {
        "Kondisi Dataset": "Sesudah Persiapan — Data Uji (20%)",
        "Jumlah Baris": f"{len(test_df):,} baris",
        "Jumlah Kolom / Fitur": "Fitur Teks Bersih + 4 Dimensi ABSA + Target 3-Kelas",
        "Keterangan": "Diisolasi ketat untuk evaluasi performa model murni (transform-only, zero leakage)."
    }
])
print("=== TABEL KONDISI DATA AKHIR SIAP PEMODELAN (BAB 3.3 LAPORAN AM552) ===")
print(summary_table.to_string(index=False))


=== PENGUJIAN PRESERVASI KATA NEGASI & NORMALISASI KONTRAKSI ===
[PASSED] Raw: "Aplikasi ini tidak bagus dan sangat mengecewakan"
         Clean: "aplikasi ini tidak bagus dan sangat mengecewakan" -> Mengandung "tidak": True
[PASSED] Raw: "Jawaban AI kurang memuaskan dan sering ngawur"
         Clean: "jawaban ai kurang memuaskan dan sering acau" -> Mengandung "kurang": True
[PASSED] Raw: "Jangan download aplikasi ini bikin emosi"
         Clean: "jangan unduh aplikasi ini membuat emosi" -> Mengandung "jangan": True
[PASSED] Raw: "Model bot bukan menjawab malah ngaco"
         Clean: "model bot bukan menjawab malah acau" -> Mengandung "bukan": True
[PASSED] Raw: "Update terbaru belum bisa login akun google"
         Clean: "update terbaru belum bisa masuk akun google" -> Mengandung "belum": True
[PASSED] Raw: "Apk nya gabisa dibuka sama sekali"
         Clean: "aplikasi nya tidak bisa dibuka sama sekali" -> Mengandung "tidak bisa": True

[OK] 100% PENGUJIAN PRESERVASI KATA NEGASI LOLOS